<a href="https://colab.research.google.com/github/SamDerden/Fraud_Detection_with-XAI_and_LLM/blob/main/%D0%94%D0%97_%D0%92%D0%B5%D0%B1%D0%B8%D0%BD%D0%B0%D1%805_%D0%90%D0%BD%D1%82%D0%B8%D1%84%D1%80%D0%BE%D0%B4_5_%D1%8D%D0%BA%D1%81%D0%BF%D0%B5%D1%80%D0%B8%D0%BC%D0%B5%D0%BD%D1%82%D0%BE%D0%B2_MLflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# ДЗ к вебинару 5: Трекинг экспериментов в MLflow (Антифрод)
# ============================================================

# --- 1. Установка и импорты ---
!pip install mlflow --quiet

import mlflow
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score, accuracy_score

print("MLflow версии", mlflow.__version__, "установлен")


# --- 2. Данные (синтетические транзакции с дисбалансом ~15%) ---
X, y = make_classification(
    n_samples=5000,
    n_features=12,
    n_informative=8,
    n_redundant=2,
    n_clusters_per_class=2,
    weights=[0.85, 0.15],
    flip_y=0.02,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Доля мошеннических транзакций в train:", round(y_train.mean(), 3))


# --- 3. Дизайн 5 экспериментов ---
# Меняем по одному гиперпараметру, чтобы честно понять вклад каждого.
mlflow.set_experiment("antifraud-homework-5runs")

experiments = [
    {
        "name": "1_baseline",
        "params": dict(n_estimators=100, max_depth=None, min_samples_leaf=1, class_weight=None)
    },
    {
        "name": "2_more_trees",
        "params": dict(n_estimators=300, max_depth=None, min_samples_leaf=1, class_weight=None)
    },
    {
        "name": "3_shallow_trees",
        "params": dict(n_estimators=100, max_depth=6, min_samples_leaf=1, class_weight=None)
    },
    {
        "name": "4_strong_leaf",
        "params": dict(n_estimators=100, max_depth=None, min_samples_leaf=10, class_weight=None)
    },
    {
        "name": "5_balanced_weight",
        "params": dict(n_estimators=100, max_depth=None, min_samples_leaf=1, class_weight="balanced")
    },
]

results = []

for exp in experiments:
    with mlflow.start_run(run_name=exp["name"]):
        model = RandomForestClassifier(
            **exp["params"],
            random_state=42,
            n_jobs=-1
        )
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        metrics = {
            "accuracy": accuracy_score(y_test, preds),
            "f1": f1_score(y_test, preds),
            "roc_auc": roc_auc_score(y_test, proba),
            "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds),
        }

        mlflow.log_params({k: str(v) for k, v in exp["params"].items()})
        mlflow.log_metrics(metrics)

        print(f"{exp['name']:20} | F1={metrics['f1']:.4f} | Recall={metrics['recall']:.4f} | ROC-AUC={metrics['roc_auc']:.4f}")

        results.append({
            "run": exp["name"],
            **exp["params"],
            **metrics
        })


# --- 4. Сводная таблица результатов ---
df = pd.DataFrame(results)
df = df.sort_values("f1", ascending=False).reset_index(drop=True)

display_cols = ["run", "n_estimators", "max_depth", "min_samples_leaf", "class_weight",
                "f1", "recall", "roc_auc", "precision"]
print("\n=== Сводная таблица ===")
display(df[display_cols].round(4))


# --- 5. Сравнение через MLflow ---
runs = mlflow.search_runs(
    experiment_names=["antifraud-homework-5runs"],
    order_by=["metrics.f1 DESC"]
)

print("\n=== Таблица из MLflow ===")
display(runs[["tags.mlflow.runName", "params.n_estimators", "params.max_depth",
              "params.min_samples_leaf", "params.class_weight",
              "metrics.f1", "metrics.recall", "metrics.roc_auc"]].round(4))


# --- 6. Вывод ---
print("""
ВЫВОД:

Мы провели 5 экспериментов, каждый раз меняя только один гиперпараметр RandomForest.

1. Самое сильное положительное влияние оказал class_weight="balanced":
   он значительно повысил Recall и F1-score относительно baseline.
   Это ожидаемо: при дисбалансе классов (~15% мошенничества) учёт весов критически важен.

2. Самое сильное отрицательное влияние оказали ограничения сложности модели:
   max_depth=6 и min_samples_leaf=10 заметно снизили F1 и особенно Recall.

3. Увеличение n_estimators с 100 до 300 почти не дало прироста качества.

ИТОГ: в задаче антифрода наиболее важным гиперпараметром оказался способ
учёта дисбаланса классов (class_weight), а не увеличение сложности/количества деревьев.
""")

MLflow версии 3.17.2 установлен
Train: (4000, 12) Test: (1000, 12)
Доля мошеннических транзакций в train: 0.156


2026/10/24 10:14:47 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/10/24 10:14:47 INFO mlflow.store.db.utils: Updating database tables
2026/10/24 10:14:50 INFO mlflow.tracking.fluent: Experiment with name 'antifraud-homework-5runs' does not exist. Creating a new experiment.


1_baseline           | F1=0.8382 | Recall=0.7308 | ROC-AUC=0.9637
2_more_trees         | F1=0.8467 | Recall=0.7436 | ROC-AUC=0.9641
3_shallow_trees      | F1=0.6971 | Recall=0.5385 | ROC-AUC=0.9397
4_strong_leaf        | F1=0.7833 | Recall=0.6603 | ROC-AUC=0.9644
5_balanced_weight    | F1=0.8266 | Recall=0.7179 | ROC-AUC=0.9682

=== Сводная таблица ===


,run,n_estimators,max_depth,min_samples_leaf,class_weight,f1,recall,roc_auc,precision
0,2_more_trees,300,NaN,1,None,0.8467,0.7436,0.9641,0.9831
1,1_baseline,100,NaN,1,None,0.8382,0.7308,0.9637,0.9828
2,5_balanced_weight,100,NaN,1,balanced,0.8266,0.7179,0.9682,0.9739
3,4_strong_leaf,100,NaN,10,None,0.7833,0.6603,0.9644,0.9626
4,3_shallow_trees,100,6.0,1,None,0.6971,0.5385,0.9397,0.9882



=== Таблица из MLflow ===


,tags.mlflow.runName,params.n_estimators,params.max_depth,params.min_samples_leaf,params.class_weight,metrics.f1,metrics.recall,metrics.roc_auc
0,2_more_trees,300,None,1,None,0.8467,0.7436,0.9641
1,1_baseline,100,None,1,None,0.8382,0.7308,0.9637
2,5_balanced_weight,100,None,1,balanced,0.8266,0.7179,0.9682
3,4_strong_leaf,100,None,10,None,0.7833,0.6603,0.9644
4,3_shallow_trees,100,6,1,None,0.6971,0.5385,0.9397



ВЫВОД:

Мы провели 5 экспериментов, каждый раз меняя только один гиперпараметр RandomForest.

1. Самое сильное положительное влияние оказал class_weight="balanced":
   он значительно повысил Recall и F1-score относительно baseline.
   Это ожидаемо: при дисбалансе классов (~15% мошенничества) учёт весов критически важен.

2. Самое сильное отрицательное влияние оказали ограничения сложности модели:
   max_depth=6 и min_samples_leaf=10 заметно снизили F1 и особенно Recall.

3. Увеличение n_estimators с 100 до 300 почти не дало прироста качества.

ИТОГ: в задаче антифрода наиболее важным гиперпараметром оказался способ
учёта дисбаланса классов (class_weight), а не увеличение сложности/количества деревьев.

